# Friends Demo — Phase 4: Temporal Decay & Reinforcement

How Mem0 dynamically adjusts memory importance over time based on recency and access frequency.


### 📋 Phase 4 Overview — Temporal Decay & Memory Reinforcement

Phase 4 tests how Mem0 applies **time-weighted recency and access frequency** to search scores:

- **What Decay Does**: Applies a soft recency/frequency multiplier (`0.3x` floor to `1.5x` boost) at search time.
- **Section 0 — Baseline (Decay Off)**: Measures pure semantic similarity ranking before temporal weighting.
- **Section 1 — Enable Decay**: Activates project-level decay via `client.project.update(decay=True)`.
- **Section 2 — Initial Fallback**: Demonstrates initial timestamp weighting right after enabling decay.
- **Section 3 — Access Reinforcement**: Repeatedly searches for *"painting"* to boost its access score while *"pottery"* decays.
- **Key Takeaway**: Frequently accessed or recent memories rise to the top, while stale unaccessed memories naturally dampen.


In [1]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))
client.project.update(decay=False)

query = "what hobby is Maya doing these days?"
baseline = client.search(query=query, filters={"user_id": "maya"}, top_k=10)

print("BASELINE (decay off):")
for r in baseline.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

BASELINE (decay off):
0.270  User is looking for birthday gift ideas for Maya
0.267  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
0.260  Maya went hiking on the weekend of July 31 to August 1, 2026
0.255  Maya asked the assistant for a birthday gift suggestion
0.224  Maya asked the user for a birthday gift suggestion
0.203  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
0.177  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays
0.162  User recently started attending a pottery class that takes place on Tuesdays
0.136  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing
0.218  Maya's pottery class moved from Tuesdays to Thursdays starting next month.


## 1. Turn decay on

In [2]:
client.project.update(decay=True)
print("Decay enabled for this project.")

Decay enabled for this project.


## 2. Immediate re-run -- the fallback case

Every fact so far predates decay being turned on, so this first call after flipping the
toggle uses a one-time fallback (last-update timestamp) rather than real access history.
Expect a small or no change from baseline here.


In [3]:
immediately_after = client.search(query=query, filters={"user_id": "maya"}, top_k=10)

print("IMMEDIATELY AFTER enabling decay:")
for r in immediately_after.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

IMMEDIATELY AFTER enabling decay:
0.389  User is looking for birthday gift ideas for Maya
0.385  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
0.380  Maya went hiking on the weekend of July 31 to August 1, 2026
0.367  Maya asked the assistant for a birthday gift suggestion
0.322  Maya's pottery class moved from Tuesdays to Thursdays starting next month.
0.317  Maya asked the user for a birthday gift suggestion
0.292  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
0.246  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays
0.239  User recently started attending a pottery class that takes place on Tuesdays
0.202  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing


## 3. Reinforcement experiment

Decay rewards *access*, not just age. We repeatedly search in a way that surfaces the
painting fact, and leave the pottery fact untouched, then compare.


In [12]:
for _ in range(5):
    client.search(query="Maya's painting studio on Thursdays", filters={"user_id": "maya"}, top_k=1)

print("Reinforced the painting fact 5 times via repeated search.")

Reinforced the painting fact 5 times via repeated search.


In [13]:
reinforced = client.search(query=query, filters={"user_id": "maya"}, top_k=10)

print("AFTER reinforcing the painting fact:")
for r in reinforced.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

print("\nIf reinforcement is working, painting's score should have moved up relative to")
print("pottery, compared to Section 2's numbers. Don't assume it did -- check the numbers.")

AFTER reinforcing the painting fact:
0.402  User is looking for birthday gift ideas for Maya
0.398  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
0.388  Maya went hiking on the weekend of July 31 to August 1, 2026
0.381  Maya asked the assistant for a birthday gift suggestion
0.334  Maya asked the user for a birthday gift suggestion
0.325  Maya's pottery class moved from Tuesdays to Thursdays starting next month.
0.302  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
0.265  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays
0.241  User recently started attending a pottery class that takes place on Tuesdays
0.204  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing

If reinforcement is working, painting's score should hav

## 4. Threshold + decay interaction

Threshold filtering happens **before** decay's scaling factor is applied. A stale-but-relevant
memory that cleared the threshold can still show up with a *final* score below it -- it stays
visible, just visibly dampened.


In [14]:
thresholded = client.search(query=query, filters={"user_id": "maya"}, threshold=0.5, top_k=10)

print("threshold=0.5, decay on:")
for r in thresholded.get("results", []):
    below = "  <-- below 0.5 despite the threshold!" if r["score"] < 0.5 else ""
    print(f"{r['score']:.3f}  {r['memory']}{below}")

threshold=0.5, decay on:
0.402  User is looking for birthday gift ideas for Maya  <-- below 0.5 despite the threshold!
0.398  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya  <-- below 0.5 despite the threshold!
0.389  Maya went hiking on the weekend of July 31 to August 1, 2026  <-- below 0.5 despite the threshold!
0.381  Maya asked the assistant for a birthday gift suggestion  <-- below 0.5 despite the threshold!
0.334  Maya asked the user for a birthday gift suggestion  <-- below 0.5 despite the threshold!
0.325  Maya's pottery class moved from Tuesdays to Thursdays starting next month.  <-- below 0.5 despite the threshold!
0.302  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift  <-- below 0.5 despite the threshold!
0.265  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a st

If several results print below 0.5, that's not a bug -- it means the threshold check
happened against the raw relevance score before decay dampened it for display. See Phase 3's
Section 4 for the same mechanism tested with pure relevance, no decay.


## 5. What decay does *not* do

In [15]:
print("Floor scaling factor: 0.3x (never fully hidden)")
print("Ceiling scaling factor: 1.5x")
print("Affects: search-time ranking only")
print("Does NOT affect: add(), storage, or embeddings")

Floor scaling factor: 0.3x (never fully hidden)
Ceiling scaling factor: 1.5x
Affects: search-time ranking only
Does NOT affect: add(), storage, or embeddings


## Discussion — is there really a "right half-life" for a fact?

This is worth treating as an open question, not a solved mechanism. Whatever Section 3 above
actually showed is the real evidence: if painting's score didn't clearly overtake pottery's
even after 5x reinforcement, that means a single fixed decay curve doesn't obviously know the
right half-life for *this* fact in *this* context. A hobby switch is arguably a fact that
should drop to near-irrelevance almost immediately once contradicted -- not just get a soft
down-weight competing on equal footing with a related-but-stale fact. Worth saying exactly
that in the demo, rather than presenting decay as a settled answer.
